# Maestro OWLv2 Object Detection Example

This notebook demonstrates how to fine-tune and run inference with the OWLv2 model in Maestro.


In [ ]:
# Setup: Install and import required libraries
%pip install transformers torch pillow --quiet

import os
import torch
from transformers import OwlViTForObjectDetection, OwlViTProcessor
from maestro.trainer.models.owlv2.core import OWLv2Configuration, train, OWLv2JSONLDataset
from maestro.trainer.models.owlv2.inference import run_inference

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Prepare dataset paths (update these as needed)
jsonl_path = '/path/to/annotations.jsonl'  # Path to your JSONL file
images_dir = '/path/to/images'  # Directory with your images

# Load processor for dataset
processor = OwlViTProcessor.from_pretrained('google/owlv2-base-patch16')

# Create dataset (for training or evaluation)
train_dataset = OWLv2JSONLDataset(jsonl_path, images_dir, processor)
print(f"Loaded {len(train_dataset)} training samples.")

In [ ]:
# Configure and train OWLv2 (similar to Florence-2 example)
config = OWLv2Configuration(
    model_name='google/owlv2-base-patch16',
    device=DEVICE,
    learning_rate=2e-5,
    epochs=5,
    batch_size=4,
    output_dir='./owlv2_output',
)

# Train the model (this may take a while)
model, processor = train(config, train_dataset)


In [ ]:
# Run inference on a new image (similar to Florence-2 example)
test_image_path = '/path/to/test_image.jpg'  # Update this path
prompts = ["cat", "dog", "car"]  # Example categories

results = run_inference(
    model_path='./owlv2_output',
    processor_path='./owlv2_output',
    image_path=test_image_path,
    prompts=prompts,
    device=DEVICE,
)

print("Inference results:")
print(results)

In [ ]:
# Visualize inference results (optional, similar to Florence-2 example)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

image = Image.open(test_image_path).convert("RGB")
plt.figure(figsize=(10, 10))
plt.imshow(image)
ax = plt.gca()

for box, score, label in zip(results['boxes'], results['scores'], results['labels']):
    xmin, ymin, xmax, ymax = box
    rect = patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin, linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)
    ax.text(xmin, ymin, f"{prompts[label]}: {score:.2f}", color='white', bbox=dict(facecolor='red', alpha=0.5))

plt.axis('off')
plt.show()